
# Detailed Solutions — Discrete Response Models Lab

Full worked solutions to every **Exercise** in `01_Extended_Lab.ipynb` and every **TODO** in
`02_Skeleton_Practice.ipynb`. Sections are numbered to match the extended lab.

Run this top to bottom; it is self-contained (re-creates all datasets it needs).


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import poisson, nbinom

pd.set_option('display.max_columns', 50)
np.random.seed(42)



---
## Part 1 Solutions — Logit / Probit


In [ ]:

train = pd.DataFrame({
    'Admitted': [1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0],
    'GPA': [2.8, 3.3, 3.7, 3.7, 3.7, 3.3, 3.7, 3, 1.7, 3.6, 3.3, 4, 3.2, 3.4, 2.8, 4, 1.5, 2.7, 2.3, 2.3, 2.7, 2.2,
            3.3, 3.3, 4, 2.3, 3.6, 3.4, 4, 3.7, 2.3],
    'Exp': [8, 6, 5, 5, 6, 3, 4, 2, 1, 5, 5, 3, 6, 5, 4, 4, 4, 1, 1, 2, 2, 2, 1, 4, 4, 4, 5, 2, 4, 6, 3]
})

logit_model = smf.logit('Admitted ~ GPA + Exp', data=train).fit()
probit_model = smf.probit('Admitted ~ GPA + Exp', data=train).fit()
print(logit_model.summary())
print(probit_model.summary())



### Solution 1.1 — Odds ratios


In [ ]:

odds_ratios = np.exp(logit_model.params)
print(odds_ratios)



**Interpretation:**
- `Intercept`: not directly meaningful (odds when GPA=0, Exp=0 — outside the data range).
- `GPA` (odds ratio ≈ 15.8): holding years of experience fixed, each 1-point increase in GPA
  multiplies the odds of admission by about 15.8x. Because GPA is on a 0–4 scale, in practice
  this is a very strong effect per fractional GPA point (e.g., +0.5 GPA roughly multiplies odds by
  $15.8^{0.5} \approx 4$).
- `Exp` (odds ratio ≈ 2.13): holding GPA fixed, each additional year of experience roughly doubles
  the odds of admission.

(Exact numbers will match whatever `logit_model.params` prints in your run — the fitted values are
deterministic given this exact dataset, but always read off your own output rather than memorizing
the numbers here.)



### Solution 1.2 — Probit vs logit coefficients

Probit coefficients are on the standard-normal z-score scale, while logit coefficients are on the
log-odds scale. The logistic distribution has variance $\pi^2/3 \approx 3.29$ versus the standard
normal's variance of 1, so logit coefficients are systematically larger in magnitude than probit
coefficients for the *same* underlying relationship — a well-known rule of thumb is
`logit_coef ≈ 1.6 to 1.8 × probit_coef`. Signs and statistical significance patterns should agree
between the two models; only the scale differs.


In [ ]:

comparison = pd.DataFrame({
    'logit': logit_model.params,
    'probit': probit_model.params,
    'ratio (logit/probit)': logit_model.params / probit_model.params
})
print(comparison)



### Solution 1.3 — Marginal effects


In [ ]:

print(logit_model.get_margeff(at='mean').summary())
print(probit_model.get_margeff(at='mean').summary())



Average marginal effects should be **much closer** between the two models than the raw coefficients,
because marginal effects re-express both models in the common units of "percentage-point change in
probability" rather than "change in log-odds" or "change in z-score."



### Solutions 1.4–1.6 — Simulated data, ROC, ground-truth recovery, pseudo-R²


In [ ]:

rng = np.random.default_rng(7)
n = 800
gpa = rng.uniform(1.5, 4.0, n)
exp_ = rng.integers(0, 10, n)
true_b0, true_bgpa, true_bexp = -11.0, 2.6, 0.7
z = true_b0 + true_bgpa * gpa + true_bexp * exp_
p = 1 / (1 + np.exp(-z))
admitted = rng.binomial(1, p)
sim = pd.DataFrame({'Admitted': admitted, 'GPA': gpa, 'Exp': exp_})

from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, ConfusionMatrixDisplay,
                              roc_curve, roc_auc_score, precision_score, recall_score, f1_score)

sim_train, sim_test = train_test_split(sim, test_size=0.25, random_state=1, stratify=sim['Admitted'])
sim_logit = smf.logit('Admitted ~ GPA + Exp', data=sim_train).fit()

y_prob = sim_logit.predict(sim_test[['GPA', 'Exp']])
y_pred = (y_prob >= 0.5).astype(int)


In [ ]:

# Solution 1.4: ROC curve
fpr, tpr, thresholds = roc_curve(sim_test['Admitted'], y_prob)
plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(sim_test['Admitted'], y_prob):.3f}")
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC curve — simulated admissions')
plt.legend(); plt.show()

print('At threshold 0.5:')
print('Precision:', precision_score(sim_test['Admitted'], y_pred))
print('Recall   :', recall_score(sim_test['Admitted'], y_pred))

# Example: if seats are limited (avoid over-admitting), raise the threshold to 0.7 to prioritize precision
y_pred_strict = (y_prob >= 0.7).astype(int)
print('\nAt threshold 0.7 (fewer, more confident admits):')
print('Precision:', precision_score(sim_test['Admitted'], y_pred_strict))
print('Recall   :', recall_score(sim_test['Admitted'], y_pred_strict))



**Discussion:** Raising the threshold from 0.5 to 0.7 trades recall for precision — fewer students
predicted "admit," but each one is a more confident prediction. A university with a hard seat cap
that wants to avoid over-offering acceptances (which historically don't all convert to enrollments)
might prefer higher precision on the "will enroll" prediction. A need-based outreach program trying
not to miss any qualified low-GPA-but-high-experience applicant might prefer higher recall (a lower
threshold) instead.


In [ ]:

# Solution 1.5: recovering the truth
print('Estimated:', sim_logit.params.values)
print('True     :', [true_b0, true_bgpa, true_bexp])



**Discussion:** With `n=800` simulated rows the MLE estimates should land reasonably close to the
true generating coefficients (small-sample bias and sampling noise mean they won't match exactly).
As $n \to \infty$, by the consistency property of maximum likelihood estimation, the estimates
converge in probability to the true parameter values, and their sampling variance shrinks — this is
why the book's 31-row / 5-row toy example should not be trusted as strongly as a well-powered study.


In [ ]:

# Solution 1.6: pseudo-R^2
print('McFadden pseudo-R2:', sim_logit.prsquared)



**Discussion:** McFadden's pseudo-$R^2 = 1 - \dfrac{\ln L_{full}}{\ln L_{null}}$ compares the fitted
model's log-likelihood to an intercept-only model's log-likelihood. It is bounded above by a number
less than 1 even for a "perfect" model in most real datasets, because it isn't measuring explained
variance the way OLS $R^2$ does — it's measuring relative improvement in fit on the log-likelihood
scale. Values of 0.2–0.4 are often considered good for this pseudo-$R^2$; you should never expect (or
demand) values near 1 the way you might for OLS.



---
## Part 2 Solutions — Multinomial Logit


In [ ]:

from sklearn import datasets
from sklearn.linear_model import LogisticRegression
import statsmodels.discrete.discrete_model as sm_discrete
from sklearn.metrics import f1_score

iris = datasets.load_iris()
df = pd.DataFrame(iris.data, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
df['target'] = iris.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)

X_train_c = sm.add_constant(X_train)
X_test_c = sm.add_constant(X_test)
model_stat = sm_discrete.MNLogit(y_train, X_train_c).fit(method='bfgs')
print(model_stat.summary())



### Solution 2.1 — Relative risk ratios


In [ ]:

rrr = np.exp(model_stat.params)
print(rrr)



**Interpretation (example):** for the class-1 (versicolor) equation, a coefficient on `petal_length`
that exponentiates to, say, 3.0 means: "holding other measurements fixed, each 1cm increase in petal
length multiplies the odds of being class 1 (versicolor) versus the baseline class 0 (setosa) by
about 3x." Always substitute your own printed coefficient value — MNLogit results depend on the
exact train/test split and solver, so read the number from your own run rather than assuming a fixed
value.



### Solution 2.2 — Regularization strength


In [ ]:

for C in [0.01, 100]:
    m = LogisticRegression(solver='lbfgs', C=C, max_iter=1000).fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f'C={C:>6}: test accuracy = {acc:.3f}, coef norm = {np.linalg.norm(m.coef_):.3f}')



**Discussion:** `C` is the inverse of the regularization strength in scikit-learn (`C = 1/lambda`).
`C=0.01` applies strong regularization, shrinking coefficients toward zero (higher bias, lower
variance) — on a clean, well-separated dataset like Iris this may barely hurt (or could even help)
accuracy. `C=100` applies almost no regularization (low bias, higher variance), which risks
overfitting on noisier or smaller datasets, though on Iris the effect is often small because the
classes are nearly linearly separable. The key takeaway: watch the coefficient magnitudes, not just
accuracy, to see regularization at work.



### Solution 2.3 — Class imbalance stress-test


In [ ]:

imbalanced = pd.concat([
    df[df.target != 2],
    df[df.target == 2].sample(frac=0.2, random_state=1)
]).reset_index(drop=True)

print(imbalanced['target'].value_counts())

Xi = imbalanced.drop('target', axis=1)
yi = imbalanced['target']
Xi_train, Xi_test, yi_train, yi_test = train_test_split(Xi, yi, test_size=0.25, random_state=1, stratify=yi)

mi = LogisticRegression(solver='lbfgs', max_iter=1000).fit(Xi_train, yi_train)
pred_i = mi.predict(Xi_test)

print('Accuracy :', accuracy_score(yi_test, pred_i))
print('Macro F1 :', f1_score(yi_test, pred_i, average='macro'))
ConfusionMatrixDisplay(confusion_matrix(yi_test, pred_i)).plot()
plt.title('Imbalanced Iris (class 2 downsampled)')
plt.show()



**Discussion:** Once class 2 is downsampled, overall accuracy can look deceptively high even if the
model rarely (or never) predicts class 2 correctly, since class 2 makes up a small share of the test
set. **Macro-F1** treats each class equally regardless of its size, so a big gap between accuracy and
macro-F1 is a red flag for imbalance-driven overconfidence.



### Solution 2.4 — 4-class / alternative dataset example


In [ ]:

from sklearn.datasets import load_wine

wine = load_wine()
Xw = pd.DataFrame(wine.data, columns=wine.feature_names)
yw = pd.Series(wine.target)

Xw_train, Xw_test, yw_train, yw_test = train_test_split(Xw, yw, test_size=0.25, random_state=1, stratify=yw)
mw = LogisticRegression(solver='lbfgs', max_iter=5000).fit(Xw_train, yw_train)
predw = mw.predict(Xw_test)

cm = confusion_matrix(yw_test, predw)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
print('Normalized confusion matrix (wine, 3 classes):')
print(np.round(cm_norm, 2))
print('Accuracy:', accuracy_score(yw_test, predw))



**Note:** the Wine dataset has 3 classes (not 4); it's used here to demonstrate the same workflow on
a different, higher-dimensional (13-feature) multinomial problem, and to show that unscaled features
of very different magnitudes (e.g., `proline` in the hundreds vs. `hue` around 1) can hurt
`LogisticRegression` convergence — in practice you'd standardize features first with
`sklearn.preprocessing.StandardScaler` for this specific dataset.



---
## Part 3 Solutions — Poisson Regression



### Solution 3.1 — Coefficient interpretation

Using the book's fitted values (Figure 8.8): `atemp` coefficient ≈ 3.0915.


In [ ]:

atemp_coef = 3.0915   # from the book's fitted Poisson model summary
multiplier = np.exp(atemp_coef)
print(f'exp(atemp coefficient) = {multiplier:.2f}')



**Interpretation:** "A one-unit increase in normalized temperature (`atemp` is scaled 0–1, so this is
effectively the full range of the variable) multiplies expected weekly casual rentals by about 22x,
holding season, weather situation, humidity, wind speed, and holiday status fixed." Because `atemp`
is normalized to [0,1], a "one-unit increase" spans the whole observed temperature range — a more
intuitive interpretation is often per 0.1-unit change: $e^{0.1 \times 3.0915} \approx 1.36$, i.e. a
36% increase in expected rentals per 0.1 increase in normalized temperature.



### Solution 3.2 — Deviance and goodness of fit

This requires fitting via `sm.GLM` (rather than `sm.Poisson`, which doesn't expose `.deviance`/`.df_resid`
the same way) to access `.deviance`. Since we may not have internet access to the bike-share data
here, we demonstrate the calculation on the offset-based insurance simulation from Exercise 3.4/3.3,
which follows the identical logic.


In [ ]:

rng = np.random.default_rng(11)
n = 500
exposure = rng.uniform(0.5, 2.0, n)
driver_age = rng.integers(18, 75, n)
prior_claims = rng.integers(0, 4, n)

log_rate = -3.0 - 0.01 * (driver_age - 40) + 0.35 * prior_claims
expected_claims = exposure * np.exp(log_rate)
claims = rng.poisson(expected_claims)

ins = pd.DataFrame({'claims': claims, 'exposure': exposure, 'driver_age': driver_age, 'prior_claims': prior_claims})
Xi = sm.add_constant(ins[['driver_age', 'prior_claims']])

glm_poisson = sm.GLM(ins['claims'], Xi, family=sm.families.Poisson(),
                      offset=np.log(ins['exposure'])).fit()
print(glm_poisson.summary())

ratio = glm_poisson.deviance / glm_poisson.df_resid
print(f'\nDeviance / df_resid = {ratio:.3f}  (close to 1 => no strong evidence of overdispersion)')



### Solution 3.3 — Offset / exposure

**When you need an offset:** whenever different observations had different "opportunity" to
accumulate the count — different observation windows, population sizes, area, or (as in the book's
bike-share example, implicitly) a different number of days averaged into a "week." Without an
offset, a row representing a big population/long time window will look like it has an inherently
higher rate than it really does, purely due to more opportunities to accumulate counts — the model
would conflate "high exposure" with "high underlying rate."

**When you don't need one:** if every observation already covers the *same* fixed window/population
(e.g., "counts per exactly 7 days" for every row), the exposure is constant and gets absorbed into
the intercept — no explicit offset term is necessary. The book's bike-share model uses *mean daily
counts per ISO week*, which sidesteps the need for an offset because every week (mostly) has the
same 7-day exposure baked into the "mean" calculation already.



### Solution 3.4 — offline offset-model fit

Already fit above as `glm_poisson`; the coefficient on `prior_claims` should come out close to the
true simulated value of 0.35, and `driver_age`'s coefficient close to -0.01, since the simulation
directly encodes those relationships (subject to sampling noise for n=500).


In [ ]:

print(glm_poisson.params)
print('\nTrue params: const≈-3.0 (before age-centering adjustment), driver_age≈-0.01, prior_claims≈0.35')



---
## Part 4 Solutions — Negative Binomial Regression


In [ ]:

data = sm.datasets.fair.load().data
data = sm.add_constant(data, prepend=False)

y = round(data['children'])
X = data[['const', 'age', 'religious', 'yrs_married', 'educ', 'occupation', 'occupation_husb', 'affairs', 'rate_marriage']]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)

poisson_model = sm.GLM(y_train, X_train, family=sm.families.Poisson()).fit()

df_aux = pd.DataFrame()
df_aux['y_mu_hat'] = poisson_model.mu
df_aux['children'] = y_train.values
df_aux['y_auxiliary'] = ((df_aux['children'] - df_aux['y_mu_hat'])**2 - df_aux['y_mu_hat']) / df_aux['y_mu_hat']
ols_model = smf.ols('y_auxiliary ~ y_mu_hat - 1', df_aux).fit()
alpha_hat = ols_model.params.iloc[0]
print(ols_model.summary())
print('\nalpha_hat =', alpha_hat)

from statsmodels.genmod.families.family import NegativeBinomial
nb_model = sm.GLM(y_train, X_train, family=NegativeBinomial(alpha=alpha_hat)).fit()
print(nb_model.summary())



### Solution 4.1 — Standard error comparison


In [ ]:

se_compare = pd.DataFrame({'poisson_se': poisson_model.bse, 'negbin_se': nb_model.bse})
se_compare['ratio (negbin/poisson)'] = se_compare['negbin_se'] / se_compare['poisson_se']
print(se_compare)



**Discussion:** Negative binomial standard errors should be **larger** than the (overconfident)
Poisson standard errors whenever real overdispersion is present — the `ratio` column typically comes
out greater than 1 for most/all coefficients. Any variable whose Poisson $p$-value was borderline
significant (e.g., 0.03–0.05) is a good candidate to check for a flip to "not significant" once the
wider negative-binomial standard errors are used.



### Solution 4.2 — Formal overdispersion test

The auxiliary regression coefficient on `y_mu_hat` is `alpha_hat`, and its associated $t$-statistic
and $p$-value (printed in the `ols_model.summary()` output above) directly test
$H_0: \alpha = 0$ (no overdispersion, i.e., Poisson is adequate) against $H_1: \alpha \neq 0$
(overdispersion is present, i.e. negative binomial is warranted). Given the book's reported estimate
(`alpha ≈ 0.622`, highly significant with $p < 0.001$ and a 95% CI of roughly [0.48, 0.76] that
excludes 0), we reject $H_0$ and conclude the data are overdispersed — negative binomial regression
is the statistically justified choice over plain Poisson for this dataset.



### Solution 4.3 — AIC/BIC comparison


In [ ]:

nb2_model = sm.NegativeBinomial(y_train, X_train).fit(disp=0)

print('Poisson AIC:', poisson_model.aic, ' BIC:', poisson_model.bic)
print('NegBin  AIC:', nb2_model.aic,   ' BIC:', nb2_model.bic)



**Discussion:** The negative binomial model should show lower (better) AIC and BIC than the Poisson
model whenever meaningful overdispersion is present, since AIC/BIC penalize the extra `alpha`
parameter but that penalty is outweighed by the improved fit. This matches the conclusion from
Exercise 4.2 — both the formal auxiliary-regression test and the AIC/BIC comparison should agree that
negative binomial is the better-supported model here.



### Solution 4.4 — Simulate overdispersed negative binomial data


In [ ]:

target_mean, target_var = 10, 40
# For NB parameterized by (n, p): mean = n(1-p)/p, var = n(1-p)/p^2
# => var/mean = 1/p  =>  p = mean/var
p_param = target_mean / target_var
n_param = target_mean**2 / (target_var - target_mean)
print('n =', n_param, ' p =', p_param)

sim_counts = nbinom.rvs(n_param, p_param, size=5000, random_state=1)
print('simulated mean:', sim_counts.mean(), ' simulated var:', sim_counts.var())

# Fit Poisson vs NegBin to this univariate (intercept-only) sample
X_const = np.ones((len(sim_counts), 1))
pois_fit = sm.GLM(sim_counts, X_const, family=sm.families.Poisson()).fit()

df_aux2 = pd.DataFrame({'mu_hat': pois_fit.mu, 'y': sim_counts})
df_aux2['y_auxiliary'] = ((df_aux2['y'] - df_aux2['mu_hat'])**2 - df_aux2['mu_hat']) / df_aux2['mu_hat']
ols2 = smf.ols('y_auxiliary ~ mu_hat - 1', df_aux2).fit()
alpha2 = ols2.params.iloc[0]
nb_fit = sm.GLM(sim_counts, X_const, family=NegativeBinomial(alpha=alpha2)).fit()

print('\nPoisson predicted mean (constant):', pois_fit.mu[0], ' SE:', pois_fit.bse[0])
print('NegBin  predicted mean (constant):', nb_fit.mu[0], ' SE:', nb_fit.bse[0])
print('\nBoth models recover the same MEAN, but only NegBin captures the true variance via alpha =', alpha2)



**Discussion:** Both models recover essentially the same estimated mean (since the mean structure is
identical), but only the negative binomial model's implied variance ($\mu + \alpha\mu^2$) matches the
simulated variance of ~40. The Poisson model implicitly assumes variance = mean = 10, understating
the true spread — this is exactly why Poisson standard errors and prediction intervals would be too
narrow for this kind of data.



---
## Part 5 Solution — Capstone


In [ ]:

def _make_mystery_data(seed=99, n=600):
    rng = np.random.default_rng(seed)
    x1 = rng.normal(0, 1, n)
    x2 = rng.integers(0, 5, n)
    mu = np.exp(0.5 + 0.8 * x1 + 0.3 * x2)
    gamma_noise = rng.gamma(shape=2.0, scale=0.5, size=n)
    y = rng.poisson(mu * gamma_noise)
    return pd.DataFrame({'y': y, 'x1': x1, 'x2': x2})

mystery = _make_mystery_data()

# Step 1: distribution
plt.hist(mystery['y'], bins=30)
plt.title('Mystery target distribution')
plt.show()

# Step 2: mean vs variance
print('mean:', mystery['y'].mean(), ' var:', mystery['y'].var(),
      ' ratio:', mystery['y'].var() / mystery['y'].mean())


In [ ]:

# Step 3: Poisson fit + dispersion check
Xm = sm.add_constant(mystery[['x1', 'x2']])
ym = mystery['y']
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.25, random_state=1)

pois_m = sm.GLM(ym_train, Xm_train, family=sm.families.Poisson()).fit()
print(pois_m.summary())
print('\nDeviance/df_resid:', pois_m.deviance / pois_m.df_resid)


In [ ]:

# Step 4: negative binomial fit + comparison
df_aux_m = pd.DataFrame({'mu_hat': pois_m.mu, 'y': ym_train.values})
df_aux_m['y_auxiliary'] = ((df_aux_m['y'] - df_aux_m['mu_hat'])**2 - df_aux_m['mu_hat']) / df_aux_m['mu_hat']
ols_m = smf.ols('y_auxiliary ~ mu_hat - 1', df_aux_m).fit()
alpha_m = ols_m.params.iloc[0]
print('alpha estimate:', alpha_m, ' p-value:', ols_m.pvalues.iloc[0])

nb_m = sm.GLM(ym_train, Xm_train, family=NegativeBinomial(alpha=alpha_m)).fit()

from sklearn.metrics import mean_squared_error
def MSE(y_true, y_pred, squared=True):
    val = mean_squared_error(y_true, y_pred)
    return val if squared else np.sqrt(val)
print('\nPoisson test RMSE:', MSE(ym_test, pois_m.predict(Xm_test), squared=False))
print('NegBin  test RMSE:', MSE(ym_test, nb_m.predict(Xm_test), squared=False))

nb2_m = sm.NegativeBinomial(ym_train, Xm_train).fit(disp=0)
print('\nPoisson AIC:', pois_m.aic, ' NegBin AIC:', nb2_m.aic)



### Step 5 — Recommendation memo

The mystery data was generated as a Poisson-Gamma mixture (a Poisson count whose rate itself has
Gamma-distributed noise) — this is, by construction, exactly the data-generating mechanism that
negative binomial regression is designed to model, and it produces variance noticeably larger than
the mean. The deviance/df_resid ratio from the Poisson fit should be well above 1, the auxiliary
regression's `alpha` should be positive and statistically significant, and the negative binomial
model should show both lower AIC and lower (or comparable) test RMSE. **Recommendation:** ship the
negative binomial model; the Poisson model's point predictions are similar, but its standard errors
and any prediction intervals built from it would understate real-world uncertainty, which matters if
this model informs decisions with financial or safety consequences (e.g., staffing or inventory
based on the count).
